# Bölüm 3: Kontrol Algoritmaları
## *Algorithms for Reinforcement Learning* — Csaba Szepesvári

Bu bölümde optimal politikayı **öğrenmeyi** hedefliyoruz. Bölüm 2'de politika sabit tutulurken değer tahmini yapıyorduk; artık politikayı da iyileştiriyoruz.

---
**Konular:**
1. Bandit Problemi ve Keşif-Sömürü İkilemi (UCB)
2. Q-Öğrenme (Tabular Q-learning)
3. Q-Öğrenme ile Fonksiyon Yaklaşımı (DQN benzeri)
4. Actor-Critic Yöntemleri
5. Politika Gradyanı (REINFORCE)


## 3.1 Kurulum ve Yardımcı Sınıflar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

class FiniteMDP:
    """Sonlu MDP simülatörü — Bölüm 3 için."""
    def __init__(self, states, actions, transitions, rewards, gamma=0.95):
        self.states       = states
        self.actions      = actions
        self.transitions  = transitions
        self.rewards      = rewards
        self.gamma        = gamma
        self.n_states     = len(states)
        self.n_actions    = len(actions)
        self.state_idx    = {s: i for i, s in enumerate(states)}
        self.action_idx   = {a: i for i, a in enumerate(actions)}
        self.current_state = states[0]
    
    def reset(self, state=None):
        self.current_state = state if state else np.random.choice(self.states)
        return self.current_state
    
    def step(self, action):
        s = self.current_state
        if (s, action) not in self.transitions:
            return s, 0.0, False
        next_states = list(self.transitions[(s, action)].keys())
        probs       = list(self.transitions[(s, action)].values())
        next_state  = np.random.choice(next_states, p=probs)
        reward      = self.rewards.get((s, action), 0.0)
        self.current_state = next_state
        done = (next_state == self.states[-1])  # Son durum terminal
        return next_state, reward, done
    
    def true_q_star(self, tol=1e-9, max_iter=2000):
        """Değer iterasyonu ile Q* hesaplar."""
        Q = np.zeros((self.n_states, self.n_actions))
        for _ in range(max_iter):
            Q_new = np.zeros_like(Q)
            for i, s in enumerate(self.states):
                for j, a in enumerate(self.actions):
                    if (s, a) not in self.transitions:
                        continue
                    q = self.rewards.get((s, a), 0)
                    for s_next, prob in self.transitions[(s, a)].items():
                        k = self.state_idx[s_next]
                        q += self.gamma * prob * np.max(Q[k])
                    Q_new[i, j] = q
            if np.max(np.abs(Q_new - Q)) < tol:
                break
            Q = Q_new
        return Q

def build_gridworld(n=4, gamma=0.95):
    """4x4 Izgara dünyası MDP."""
    states  = list(range(n * n))
    actions = ['up', 'down', 'left', 'right']
    goal, trap = n*n - 1, n*n - 5
    
    def move(s, a):
        r, c = s // n, s % n
        if a == 'up'    and r > 0:   return s - n
        if a == 'down'  and r < n-1: return s + n
        if a == 'left'  and c > 0:   return s - 1
        if a == 'right' and c < n-1: return s + 1
        return s
    
    transitions, rewards = {}, {}
    for s in states:
        for a in actions:
            if s in [goal, trap]:
                transitions[(s, a)] = {s: 1.0}
                rewards[(s, a)]     = 0.0
                continue
            ns_main = move(s, a)
            others  = [move(s, oa) for oa in actions if oa != a]
            trans   = {ns_main: 0.7}
            for ns_s in others:
                trans[ns_s] = trans.get(ns_s, 0) + 0.1
            transitions[(s, a)] = trans
            rewards[(s, a)]     = (1.0 if ns_main == goal else
                                   (-1.0 if ns_main == trap else -0.01))
    return FiniteMDP(states, actions, transitions, rewards, gamma), goal, trap

env, GOAL, TRAP = build_gridworld()
Q_star = env.true_q_star()
V_star = np.max(Q_star, axis=1)
print(f"Izgara dünyası hazır. Q* boyutu: {Q_star.shape}")
print(f"Hedef durum: {GOAL}, Tuzak: {TRAP}")

## 3.2 Bandit Problemi: Keşif-Sömürü İkilemi

Çok kollu bandit problemi, pekiştirmeli öğrenmedeki **exploration-exploitation** (keşif-sömürü) ikileminin özüdür.

**Güven Aralığı Üst Sınırı (UCB):** Szepesvári s.38-40
$$A_t = \arg\max_a \left[ \hat{\mu}_a + \sqrt{\frac{2 \ln t}{N_t(a)}} \right]$$

Burada:
- $\hat{\mu}_a$: $a$ kolunun ortalama tahmini ödülü
- $N_t(a)$: $a$ kolunun seçilme sayısı
- İkinci terim: güven üst sınırı (bonus)


In [ ]:
class MultiArmedBandit:
    """K-kollu stokastik bandit."""
    def __init__(self, means, stds=None):
        self.K     = len(means)
        self.means = np.array(means)
        self.stds  = np.ones(self.K) if stds is None else np.array(stds)
        self.optimal_arm = np.argmax(self.means)
    
    def pull(self, arm):
        return np.random.normal(self.means[arm], self.stds[arm])

# ── Algoritmalar ──────────────────────────────────────────────────

def epsilon_greedy(bandit, n_steps, epsilon):
    """ε-greedy politikası"""
    Q = np.zeros(bandit.K)
    N = np.zeros(bandit.K)
    rewards, regrets = [], []
    
    for t in range(n_steps):
        if np.random.rand() < epsilon:
            arm = np.random.randint(bandit.K)
        else:
            arm = np.argmax(Q)
        
        r = bandit.pull(arm)
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    
    return np.array(rewards), np.cumsum(regrets)

def ucb1(bandit, n_steps, c=2.0):
    """UCB1 — Szepesvári Algoritma 5 (s.39)"""
    Q = np.zeros(bandit.K)
    N = np.zeros(bandit.K)
    rewards, regrets = [], []
    
    # Her kolu bir kez çek
    for arm in range(bandit.K):
        r = bandit.pull(arm)
        N[arm] = 1
        Q[arm] = r
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    
    for t in range(bandit.K, n_steps):
        # UCB skoru
        ucb_scores = Q + np.sqrt(c * np.log(t + 1) / (N + 1e-9))
        arm = np.argmax(ucb_scores)
        
        r = bandit.pull(arm)
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    
    return np.array(rewards), np.cumsum(regrets)

def thompson_sampling(bandit, n_steps):
    """Thompson Sampling (Bayes yaklaşım)"""
    # Her kol için Beta(alpha, beta) — Bernoulli ödül varsayımı
    # Normal ödüller için normalleştiriyoruz
    Q = np.zeros(bandit.K)
    N = np.zeros(bandit.K)
    rewards, regrets = [], []
    
    for t in range(n_steps):
        # Posterior örnekleme
        samples = np.random.normal(Q, 1.0 / (N + 1))
        arm = np.argmax(samples)
        
        r = bandit.pull(arm)
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    
    return np.array(rewards), np.cumsum(regrets)

# ── Simülasyon ────────────────────────────────────────────────────
np.random.seed(42)
K, T, n_runs = 10, 2000, 30
true_means  = np.random.normal(0, 1, K)
print(f"K={K} kollu bandit. Optimal kol: {np.argmax(true_means)} (μ*={true_means.max():.3f})")

# n_runs simülasyon ortalaması
def run_bandit(algo_fn, n_runs=30):
    all_regrets = []
    for _ in range(n_runs):
        bandit = MultiArmedBandit(true_means)
        _, cumreg = algo_fn(bandit, T)
        all_regrets.append(cumreg)
    return np.mean(all_regrets, axis=0), np.std(all_regrets, axis=0)

reg_eps_mean, reg_eps_std = run_bandit(lambda b,t: epsilon_greedy(b, t, 0.1))
reg_ucb_mean, reg_ucb_std = run_bandit(lambda b,t: ucb1(b, t, c=2.0))
reg_ts_mean,  reg_ts_std  = run_bandit(lambda b,t: thompson_sampling(b, t))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

t_ax = np.arange(T)
for mean, std, label, color in [
    (reg_eps_mean, reg_eps_std, 'ε-greedy (ε=0.1)', 'red'),
    (reg_ucb_mean, reg_ucb_std, 'UCB1 (c=2)',        'steelblue'),
    (reg_ts_mean,  reg_ts_std,  'Thompson Sampling',  'green'),
]:
    axes[0].plot(t_ax, mean, color=color, linewidth=2, label=label)
    axes[0].fill_between(t_ax, mean-std, mean+std, color=color, alpha=0.15)

axes[0].set_xlabel('Adım t')
axes[0].set_ylabel('Kümülatif Pişmanlık (Regret)')
axes[0].set_title(f'Bandit Algoritmaları Karşılaştırması\n({n_runs} çalışma ortalaması)', fontweight='bold')
axes[0].legend()

# UCB bonuslarını görselleştir
N_demo   = np.array([1, 1, 5, 10, 50, 100])
t_demo   = 1000
ucb_bonus = np.sqrt(2 * np.log(t_demo) / N_demo)
axes[1].bar(range(len(N_demo)), ucb_bonus, color='steelblue', alpha=0.8)
axes[1].set_xticks(range(len(N_demo)))
axes[1].set_xticklabels([f'N={n}' for n in N_demo])
axes[1].set_ylabel('UCB Bonusu')
axes[1].set_title(f'UCB Keşif Bonusu (t={t_demo})', fontweight='bold')
axes[1].set_xlabel('Kolun kaç kez çekildiği (N)')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_bandit.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final regret → ε-greedy: {reg_eps_mean[-1]:.1f}  UCB1: {reg_ucb_mean[-1]:.1f}  TS: {reg_ts_mean[-1]:.1f}")

## 3.3 Q-Öğrenme (Tabular Q-Learning)

### Algoritma 12 (Szepesvári, s.47)

```
function QLearning(X, A, R, Y, Q):
    δ ← R + γ·max_{a'} Q[Y,a'] − Q[X,A]
    Q[X,A] ← Q[X,A] + α·δ
    return Q
```

**Güncelleme denklemi:**
$$Q_{t+1}(x, a) = Q_t(x, a) + \alpha_t \left[ R_{t+1} + \gamma \max_{a'} Q_t(X_{t+1}, a') - Q_t(x, a) \right]$$

Q-öğrenme **off-policy**'dir: davranış politikasından bağımsız olarak Q*'a yakınsar.

**Yakınsama garantisi:** Her durum-eylem çifti sonsuz kez ziyaret edildiğinde ve Robbins-Monro koşulları sağlandığında $Q_t \to Q^*$ (Tsitsiklis 1994).


In [ ]:
def q_learning(env, n_episodes=3_000, alpha=0.1, epsilon=0.2, 
               epsilon_decay=True, verbose=True):
    """
    Tabular Q-Öğrenme — Algoritma 12 (Szepesvári)
    
    epsilon_decay: ε'yi adım sayısına göre azalt
    """
    Q = np.zeros((env.n_states, env.n_actions))
    _Q_star = env.true_q_star()        # Bu env için Q* hesapla
    eps_history, rmse_history = [], []
    
    for ep in range(n_episodes):
        state = env.reset(state=env.states[0])
        eps   = epsilon / (1 + ep * 0.001) if epsilon_decay else epsilon
        total_reward = 0
        
        for step in range(200):
            # ε-greedy eylem seçimi
            if np.random.rand() < eps:
                action_idx = np.random.randint(env.n_actions)
            else:
                i = env.state_idx[state]
                action_idx = np.argmax(Q[i])
            
            action = env.actions[action_idx]
            next_state, reward, done = env.step(action)
            
            # Q-güncelleme
            i  = env.state_idx[state]
            j  = env.state_idx[next_state]
            td = reward + env.gamma * np.max(Q[j]) - Q[i, action_idx]
            Q[i, action_idx] += alpha * td
            
            total_reward += reward
            state = next_state
            if done:
                break
        
        if ep % 50 == 0:
            rmse = np.sqrt(np.mean((Q - _Q_star) ** 2))
            rmse_history.append(rmse)
            eps_history.append(eps)
    
    return Q, rmse_history, eps_history

# Farklı parametreler
print("Q-Öğrenme çalışıyor...")
Q_est, rmse_hist, eps_hist = q_learning(env, n_episodes=5_000, alpha=0.1, 
                                         epsilon=0.5, epsilon_decay=True)

# Optimal politika karşılaştırması
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Q değerleri ısı haritası
n = env.n_states
im = axes[0].imshow(Q_est, aspect='auto', cmap='viridis')
axes[0].set_xlabel('Eylem indeksi')
axes[0].set_ylabel('Durum indeksi')
axes[0].set_title('Q-Öğrenme: Q(s,a) Tahmini', fontweight='bold')
axes[0].set_xticks(range(env.n_actions))
axes[0].set_xticklabels(env.actions, rotation=45)
plt.colorbar(im, ax=axes[0])

# 2. RMSE yakınsama
ep_ticks = np.arange(len(rmse_hist)) * 50
axes[1].plot(ep_ticks, rmse_hist, 'steelblue', linewidth=2)
axes[1].set_xlabel('Epizot')
axes[1].set_ylabel('RMSE(Q, Q*)')
axes[1].set_title('Q-Öğrenme Yakınsama (Q → Q*)', fontweight='bold')
axes[1].fill_between(ep_ticks, rmse_hist, alpha=0.2)

# 3. Öğrenilen politika (GridWorld)
n_grid = 4
arrow_map = {'up': '↑', 'down': '↓', 'left': '←', 'right': '→'}
axes[2].set_xlim(-0.5, n_grid - 0.5)
axes[2].set_ylim(-0.5, n_grid - 0.5)
axes[2].set_title('Öğrenilen Optimal Politika', fontweight='bold')

for s in env.states:
    i_s = env.state_idx[s]
    row, col = s // n_grid, s % n_grid
    if s == GOAL:
        axes[2].text(col, n_grid-1-row, 'G', ha='center', va='center',
                    fontsize=16, color='green', fontweight='bold')
    elif s == TRAP:
        axes[2].text(col, n_grid-1-row, '✗', ha='center', va='center',
                    fontsize=16, color='red', fontweight='bold')
    else:
        best_a = env.actions[np.argmax(Q_est[i_s])]
        axes[2].text(col, n_grid-1-row, arrow_map[best_a],
                    ha='center', va='center', fontsize=18)

axes[2].set_xticks(range(n_grid))
axes[2].set_yticks(range(n_grid))
axes[2].grid(True, alpha=0.5)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_qlearning.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final RMSE: {rmse_hist[-1]:.5f}")

## 3.4 SARSA (On-Policy TD Kontrolü)

Q-öğrenme **off-policy** iken, SARSA **on-policy**'dir: davranış politikasıyla aynı politikayı öğrenir.

**Güncelleme:**
$$Q_{t+1}(X_t, A_t) = Q_t(X_t, A_t) + \alpha \left[ R_{t+1} + \gamma Q_t(X_{t+1}, A_{t+1}) - Q_t(X_t, A_t) \right]$$

**On vs Off-policy fark:** SARSA bir sonraki adımın *gerçekte seçilen* eylemini kullanırken, Q-öğrenme *maksimum* değeri kullanır.


In [ ]:
def sarsa(env, n_episodes=3_000, alpha=0.1, epsilon=0.5, epsilon_decay=True):
    """
    SARSA — On-policy TD kontrolü
    """
    Q      = np.zeros((env.n_states, env.n_actions))
    Q_opt  = env.true_q_star()          # Bu env'in Q* değeri
    rmse_history = []

    def eps_greedy(state, eps):
        i = env.state_idx[state]
        if np.random.rand() < eps:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[i])

    for ep in range(n_episodes):
        eps   = epsilon / (1 + ep * 0.001) if epsilon_decay else epsilon
        state = env.reset(state=env.states[0])
        a_idx = eps_greedy(state, eps)

        for _ in range(200):
            action     = env.actions[a_idx]
            next_state, reward, done = env.step(action)
            next_a_idx = eps_greedy(next_state, eps)

            i  = env.state_idx[state]
            j  = env.state_idx[next_state]
            td = reward + env.gamma * Q[j, next_a_idx] - Q[i, a_idx]
            Q[i, a_idx] += alpha * td

            state, a_idx = next_state, next_a_idx
            if done:
                break

        if ep % 50 == 0:
            rmse = np.sqrt(np.mean((Q - Q_opt) ** 2))
            rmse_history.append(rmse)

    return Q, rmse_history


# Karşılaştırma: Q-learning vs SARSA
print("Q-learning ve SARSA karşılaştırılıyor...")
n_runs_ctrl = 10   # hız için azaltıldı

def run_ctrl(algo_fn, n_runs=10):
    """algo_fn(env) → rmse_history listesi döner"""
    all_rmse = []
    for _ in range(n_runs):
        env2, _, _ = build_gridworld()
        Q_opt2     = env2.true_q_star()

        if algo_fn == q_learning:
            Q_res, rmse, _ = q_learning(env2, n_episodes=3_000,
                                         alpha=0.1, epsilon=0.5, verbose=False)
        else:
            Q_res, rmse = sarsa(env2, n_episodes=3_000, alpha=0.1, epsilon=0.5)

        all_rmse.append(rmse)
    arr = np.array(all_rmse)
    return arr.mean(0), arr.std(0)

ql_mean, ql_std = run_ctrl(q_learning, n_runs=n_runs_ctrl)
sa_mean, sa_std = run_ctrl(sarsa,      n_runs=n_runs_ctrl)

fig, ax = plt.subplots(figsize=(10, 5))
ep_ticks = np.arange(len(ql_mean)) * 50

ax.plot(ep_ticks, ql_mean, 'steelblue',  linewidth=2.5, label='Q-Learning (off-policy)')
ax.fill_between(ep_ticks, ql_mean-ql_std, ql_mean+ql_std, color='steelblue', alpha=0.2)
ax.plot(ep_ticks, sa_mean, 'darkorange', linewidth=2.5, label='SARSA (on-policy)')
ax.fill_between(ep_ticks, sa_mean-sa_std, sa_mean+sa_std, color='darkorange', alpha=0.2)

ax.set_xlabel('Epizot')
ax.set_ylabel('RMSE(Q, Q*)')
ax.set_title(f'Q-Learning vs SARSA ({n_runs_ctrl} çalışma ortalaması, gölge=std)', fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_sarsa_vs_ql.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.5 Actor-Critic Yöntemleri

Actor-Critic, Szepesvári'nin Bölüm 3.4'ünde açıklanan temel yapıdır:

- **Actor (Aktör):** Politikayı $\pi_\theta$ parametrize eder ve iyileştirir.
- **Critic (Eleştirmen):** Değer fonksiyonunu $V_w$ tahmin eder ve aktöre sinyal verir.

### Güncelleme (Bölüm 3.4.1-3.4.2, s.54-60)

**Critic (TD hatası):**
$$\delta_t = R_{t+1} + \gamma V_w(X_{t+1}) - V_w(X_t)$$
$$w \leftarrow w + \alpha_c \delta_t \nabla_w V_w(X_t)$$

**Actor (Politika gradyanı):**
$$\theta \leftarrow \theta + \alpha_a \delta_t \nabla_\theta \log \pi_\theta(A_t \mid X_t)$$

**Softmax politika:**
$$\pi_\theta(a \mid s) = \frac{e^{\theta_{s,a}}}{\sum_{a'} e^{\theta_{s,a'}}}$$


In [ ]:
def actor_critic(env, n_episodes=4_000, alpha_actor=0.01, alpha_critic=0.1, gamma=0.95):
    """
    Tabular Actor-Critic — Szepesvári Bölüm 3.4
    
    Actor:   θ[s, a] - politika parametreleri (softmax)
    Critic:  w[s]    - değer fonksiyonu ağırlıkları
    """
    _Q_ac  = env.true_q_star()
    _V_star = np.max(_Q_ac, axis=1)                   # Referans V*
    theta = np.zeros((env.n_states, env.n_actions))   # Actor
    w     = np.zeros(env.n_states)                    # Critic
    
    def softmax_policy(s):
        i     = env.state_idx[s]
        prefs = theta[i] - np.max(theta[i])  # Sayısal kararlılık
        probs = np.exp(prefs)
        return probs / probs.sum()
    
    rewards_hist, rmse_hist = [], []
    
    for ep in range(n_episodes):
        state        = env.reset(state=env.states[0])
        total_reward = 0
        
        for step in range(300):
            # Softmax politikasından eylem örnekle
            probs      = softmax_policy(state)
            action_idx = np.random.choice(env.n_actions, p=probs)
            action     = env.actions[action_idx]
            
            next_state, reward, done = env.step(action)
            
            i = env.state_idx[state]
            j = env.state_idx[next_state]
            
            # TD hatası (Critic)
            delta = reward + gamma * w[j] * (1 - done) - w[i]
            
            # Critic güncellemesi
            w[i] += alpha_critic * delta
            
            # Actor güncellemesi: ∇_θ log π_θ(a|s)
            # Softmax gradyanı: I[a=A] - π(a|s)
            grad_log_pi      = -probs.copy()
            grad_log_pi[action_idx] += 1.0
            theta[i]        += alpha_actor * delta * grad_log_pi
            
            total_reward += reward
            state = next_state
            if done:
                break
        
        rewards_hist.append(total_reward)
        if ep % 50 == 0:
            # Policy değer fonksiyonu
            V_est = w
            rmse_hist.append(np.sqrt(np.mean((V_est - _V_star) ** 2)))
    
    return theta, w, rewards_hist, rmse_hist

print("Actor-Critic eğitiliyor...")
theta_ac, w_ac, rew_hist, rmse_ac = actor_critic(env, n_episodes=5_000,
                                                    alpha_actor=0.02, alpha_critic=0.2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Öğrenilen politika
n_grid    = 4
arrow_map = {'up': '↑', 'down': '↓', 'left': '←', 'right': '→'}
axes[0].set_title('Actor-Critic Öğrenilen Politika', fontweight='bold')
axes[0].set_xlim(-0.5, n_grid-0.5)
axes[0].set_ylim(-0.5, n_grid-0.5)

for s in env.states:
    i_s   = env.state_idx[s]
    row, col = s // n_grid, s % n_grid
    prefs = theta_ac[i_s] - np.max(theta_ac[i_s])
    probs = np.exp(prefs) / np.exp(prefs).sum()
    best_a_idx = np.argmax(probs)
    
    if s == GOAL:
        axes[0].text(col, n_grid-1-row, 'G', ha='center', va='center',
                    fontsize=16, color='green', fontweight='bold')
    elif s == TRAP:
        axes[0].text(col, n_grid-1-row, '✗', ha='center', va='center',
                    fontsize=16, color='red', fontweight='bold')
    else:
        axes[0].text(col, n_grid-1-row, arrow_map[env.actions[best_a_idx]],
                    ha='center', va='center', fontsize=18)
        
axes[0].set_xticks(range(n_grid))
axes[0].set_yticks(range(n_grid))
axes[0].grid(True, alpha=0.5)

# 2. Eğitim ödülü
window = 100
smoothed = np.convolve(rew_hist, np.ones(window)/window, mode='valid')
axes[1].plot(rew_hist, alpha=0.3, color='steelblue', linewidth=0.8)
axes[1].plot(smoothed, 'steelblue', linewidth=2.5, label=f'Hareketli ortalama (w={window})')
axes[1].set_xlabel('Epizot')
axes[1].set_ylabel('Toplam Ödül')
axes[1].set_title('Actor-Critic: Epizot Ödülleri', fontweight='bold')
axes[1].legend()

# 3. Critic değer tahmini
x_states = np.arange(env.n_states)
axes[2].plot(x_states, V_star, 'k-', linewidth=2.5, label='V* (Gerçek)', zorder=5)
axes[2].plot(x_states, w_ac,   'r--', linewidth=2, label='Critic V̂')
axes[2].set_xlabel('Durum indeksi')
axes[2].set_ylabel('Değer')
axes[2].set_title('Critic: Değer Fonksiyonu Tahmini', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_actor_critic.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 REINFORCE: Politika Gradyanı

REINFORCE (Williams 1992), Monte Carlo geriye dönük tahminle politika gradyanını hesaplar:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[ G_t \nabla_\theta \log \pi_\theta(A_t \mid X_t) \right]$$

**Güncelleme (her adım için):**
$$\theta \leftarrow \theta + \alpha G_t \nabla_\theta \log \pi_\theta(A_t \mid X_t)$$

**Szepesvári Bölüm 3.4.2'ye göre:** Aktör, politika gradyanı yöntemlerinden biri olarak uygulanır.


In [ ]:
def reinforce(env, n_episodes=4_000, alpha=0.01, gamma=0.95, baseline=True):
    """
    REINFORCE Politika Gradyanı
    baseline=True: Varyans azaltma için ortalama getiri çıkarılır
    """
    theta    = np.zeros((env.n_states, env.n_actions))
    baseline_val = np.zeros(env.n_states)  # Durum değer tabanı
    
    def softmax_policy(s):
        i     = env.state_idx[s]
        prefs = theta[i] - np.max(theta[i])
        probs = np.exp(prefs)
        return probs / probs.sum()
    
    def generate_episode_policy(max_steps=300):
        state  = env.reset(state=env.states[0])
        traj   = []
        for _ in range(max_steps):
            probs      = softmax_policy(state)
            a_idx      = np.random.choice(env.n_actions, p=probs)
            action     = env.actions[a_idx]
            next_s, r, done = env.step(action)
            traj.append((state, a_idx, r))
            state = next_s
            if done:
                break
        return traj
    
    rewards_hist = []
    
    for ep in range(n_episodes):
        traj   = generate_episode_policy()
        T      = len(traj)
        
        # Getiri hesapla (geriye doğru)
        returns = np.zeros(T)
        G = 0
        for t in range(T-1, -1, -1):
            G         = traj[t][2] + gamma * G
            returns[t] = G
        
        # Güncelleme
        for t, (s, a_idx, r) in enumerate(traj):
            i     = env.state_idx[s]
            probs = softmax_policy(s)
            G_t   = returns[t]
            
            if baseline:
                G_t -= baseline_val[i]
                # Baseline güncelle (ortalama getiri)
                baseline_val[i] += 0.01 * (returns[t] - baseline_val[i])
            
            grad_log_pi          = -probs.copy()
            grad_log_pi[a_idx]  += 1.0
            theta[i]            += alpha * G_t * grad_log_pi
        
        rewards_hist.append(sum(r for _, _, r in traj))
    
    return theta, rewards_hist

print("REINFORCE eğitiliyor (baseline ile ve baseline olmadan)...")
theta_rf,    rh_rf    = reinforce(env, n_episodes=3_000, alpha=0.02, baseline=False)
theta_rf_bl, rh_rf_bl = reinforce(env, n_episodes=3_000, alpha=0.02, baseline=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

window = 100
for rh, label, color in [
    (rh_rf,    'REINFORCE',             'darkorange'),
    (rh_rf_bl, 'REINFORCE + Baseline',  'steelblue'),
]:
    axes[0].plot(rh, alpha=0.2, color=color)
    smoothed = np.convolve(rh, np.ones(window)/window, mode='valid')
    axes[0].plot(smoothed, color=color, linewidth=2.5, label=label)

axes[0].set_xlabel('Epizot')
axes[0].set_ylabel('Toplam Ödül')
axes[0].set_title('REINFORCE vs REINFORCE + Baseline', fontweight='bold')
axes[0].legend()

# Politika karşılaştırması
methods = ['Q-Learning', 'Actor-Critic', 'REINFORCE+BL']
thetas  = [None, theta_ac, theta_rf_bl]
Qs_list = [Q_est, None, None]

# Her yöntem için V tahmin et
def policy_values(theta_or_Q, is_Q=False):
    V_est = np.zeros(env.n_states)
    for s in env.states:
        i = env.state_idx[s]
        if is_Q:
            V_est[i] = np.max(theta_or_Q[i])
        else:
            prefs = theta_or_Q[i] - np.max(theta_or_Q[i])
            probs = np.exp(prefs) / np.exp(prefs).sum()
            V_est[i] = np.sum(probs * theta_or_Q[i])  # approximation
    return V_est

V_q  = np.max(Q_est, axis=1)
V_ac = w_ac
V_rf = np.max(theta_rf_bl, axis=1)

axes[1].plot(env.states, V_star, 'k-',  linewidth=3, label='V* (Optimal)', zorder=5)
axes[1].plot(env.states, V_q,    'b--', linewidth=2, label='Q-Learning V')
axes[1].plot(env.states, V_ac,   'r-.',  linewidth=2, label='Actor-Critic V')
axes[1].set_xlabel('Durum indeksi')
axes[1].set_ylabel('Değer Tahmini')
axes[1].set_title('Yöntemler Arası Değer Fonksiyonu Karşılaştırması', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_reinforce.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7 Tüm Algoritmaları Karşılaştır: Öğrenme Hızı

Şimdiye kadar gördüğümüz tüm kontrol algoritmalarını performans açısından karşılaştıralım.


In [ ]:
# ── Kapsamlı kontrol algoritmaları karşılaştırması ───────────────
print("Kapsamlı karşılaştırma başlıyor...")
N_EPISODES = 2_000   # hız için azaltıldı
N_RUNS     = 5       # hız için azaltıldı

def avg_reward_curve(algo_fn, n_runs=N_RUNS):
    """Çoklu çalışma ortalaması alır."""
    all_curves = []
    for _ in range(n_runs):
        env_tmp, _, _ = build_gridworld()
        curve = algo_fn(env_tmp)
        all_curves.append(curve)
    min_len = min(len(c) for c in all_curves)
    arr = np.array([c[:min_len] for c in all_curves])
    return arr.mean(0), arr.std(0)

def ql_curve(env_tmp):
    _, rmse, _ = q_learning(env_tmp, n_episodes=N_EPISODES,
                             alpha=0.1, epsilon=0.5, epsilon_decay=True, verbose=False)
    return rmse

def sarsa_curve(env_tmp):
    _, rmse = sarsa(env_tmp, n_episodes=N_EPISODES, alpha=0.1, epsilon=0.5)
    return rmse

def ac_curve(env_tmp):
    _, _, _, rmse = actor_critic(env_tmp, n_episodes=N_EPISODES,
                                  alpha_actor=0.02, alpha_critic=0.2)
    return rmse

curves = {
    'Q-Learning':   avg_reward_curve(ql_curve),
    'SARSA':        avg_reward_curve(sarsa_curve),
    'Actor-Critic': avg_reward_curve(ac_curve),
}

fig, ax = plt.subplots(figsize=(12, 6))
colors = {'Q-Learning': 'steelblue', 'SARSA': 'darkorange', 'Actor-Critic': 'green'}

for name, (mean, std) in curves.items():
    ep_ticks = np.arange(len(mean)) * 50
    ax.plot(ep_ticks, mean, color=colors[name], linewidth=2.5, label=name)
    ax.fill_between(ep_ticks, mean-std, mean+std, color=colors[name], alpha=0.15)

ax.set_xlabel('Epizot', fontsize=13)
ax.set_ylabel('RMSE (Q veya V → Optimal)', fontsize=13)
ax.set_title(f'Kontrol Algoritmaları Karşılaştırması\n({N_RUNS} çalışma ortalaması, gölge = std)',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n─── Final RMSE Özeti ───")
for name, (mean, std) in curves.items():
    print(f"{name:<20}: {mean[-1]:.5f} ± {std[-1]:.5f}")


## 3.8 Özet

| Algoritma | On/Off-policy | Model | Yakınsama | Ölçekleme |
|-----------|--------------|-------|-----------|-----------|
| **Q-Learning**  | Off-policy | Model-free | Q* | Tabular |
| **SARSA**       | On-policy  | Model-free | $Q^{\pi_{\epsilon}}$ | Tabular |
| **Actor-Critic** | On-policy | Model-free | Yerel optimum | FA ile büyür |
| **REINFORCE**   | On-policy  | Model-free | Yerel optimum | FA ile büyür |
| **UCB**         | —          | Model-free | Regret min. | Bandit |

### Kilit Kavramlar
- **Off-policy:** Davranış politikası ≠ hedef politika (Q-learning)
- **On-policy:** Davranış politikası = hedef politika (SARSA, Actor-Critic)
- **Exploration-Exploitation:** UCB prensibi optimal keşfi sağlar
- **Politika Gradyanı:** Doğrudan $\theta$ optimize eder; değer fonksiyonu aracı

### Pratik Tavsiyeler
1. Küçük, sonlu ortamlar → **Q-Learning** veya **SARSA**
2. Sürekli durum/eylem → **Actor-Critic** + fonksiyon yaklaşımı
3. Bandit problemi → **UCB1** veya **Thompson Sampling**
4. Yüksek varyans → **Baseline** (REINFORCE + Baseline)
